# 02 — Pruning the basket & regime detection

**Phase 2, continued.** Two jobs:

1. **Prune.** The monthly panel now carries ~96 columns (core prices + 74 Pink
   Sheet series). Most are noise for a macro dashboard. Cluster the basket by
   return correlation and keep only series that show real structure — either
   with the macro layer or with the core metals/energy complex.
2. **Detect regimes.** Operationalize the earlier finding (the gold–real-yield
   link is *state-dependent*) into explicit volatility regimes the dashboard can
   flag, using a rolling-volatility state split and a changepoint check.

**Discipline unchanged:** explore window only. Holdout is never loaded here.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))
import config

pd.set_option("display.width", 140)
plt.rcParams["figure.figsize"] = (11, 5)

## Load monthly, cut to explore

In [ ]:
monthly = pd.read_parquet(config.DATA / "commodities_monthly.parquet")
monthly_ex = monthly.loc[:config.EXPLORE_END].copy()
print("monthly explore:", monthly_ex.shape)

# separate the three column families
core_prices = ["gold","silver","wti","brent","natgas","copper","palladium","platinum"]
macro_cols  = ["usd_broad","ust10y_nominal","ust10y_real","vix","fed_funds"]
pink_cols   = [c for c in monthly_ex.columns if c.startswith("pink_")]
print(f"core={len(core_prices)}  macro={len(macro_cols)}  pink={len(pink_cols)}")

## Step 1 — Prune the Pink Sheet basket

Compute monthly log-returns for every Pink Sheet series, then score each on two
axes: (a) does it move with the macro layer, (b) does it move with the core
complex. A series that does neither is not worth carrying.

In [ ]:
# log returns; guard against non-positive values in some Pink series
def safe_logret(df):
    d = df.where(df > 0)
    return np.log(d).diff()

pink_ret = safe_logret(monthly_ex[pink_cols])
core_ret = safe_logret(monthly_ex[core_prices])
macro_chg = monthly_ex[macro_cols].diff()

# require enough history to be meaningful
valid = pink_ret.columns[pink_ret.notna().sum() >= 120]  # >=10y of months
pink_ret = pink_ret[valid]
print(f"pink series with >=120 obs: {len(valid)} of {len(pink_cols)}")

In [ ]:
# score each pink series: max |corr| vs macro, and max |corr| vs core
rows = []
for col in pink_ret.columns:
    s = pink_ret[col]
    macro_score = max(abs(s.corr(macro_chg[m])) for m in macro_cols)
    core_score  = max(abs(s.corr(core_ret[p]))  for p in core_prices)
    rows.append({"series": col, "macro": macro_score, "core": core_score,
                 "best": max(macro_score, core_score)})
score = pd.DataFrame(rows).set_index("series").sort_values("best", ascending=False)
score.round(2).head(25)

**Keep rule:** a series earns a place if it correlates at least ~0.3 with
*something* structural (macro or core). Below that it's idiosyncratic noise for
our purposes. Adjust the threshold once you see the distribution.

In [ ]:
THRESH = 0.30
keep_pink = score.index[score["best"] >= THRESH].tolist()
drop_pink = score.index[score["best"] < THRESH].tolist()
print(f"KEEP {len(keep_pink)} pink series:")
for s in keep_pink: print("  ", s, round(score.loc[s,"best"],2))
print(f"\nDROP {len(drop_pink)} (idiosyncratic for our purposes)")

# the working basket the rest of the project uses
basket = core_prices + keep_pink
print(f"\nworking basket: {len(basket)} series")

## Step 2 — Regime detection

Define a market-wide volatility state from the core complex, then show that the
gold–real-yield relationship differs by state. This turns the earlier rolling
chart into a discrete, dashboard-usable flag.

In [ ]:
# market vol proxy: cross-sectional avg of rolling 12m stdev of core returns
core_vol = core_ret.rolling(12).std().mean(axis=1)
# regime = high vol when above its own 70th percentile (explore-derived cutoff)
cutoff = core_vol.quantile(0.70)
regime = (core_vol > cutoff).map({True: "high_vol", False: "calm"})
print("cutoff (70th pct of avg 12m vol):", round(cutoff, 4))
print(regime.value_counts())

fig, ax = plt.subplots()
core_vol.plot(ax=ax, color="steelblue", label="avg 12m vol (core)")
ax.axhline(cutoff, color="darkred", ls="--", lw=1, label="high-vol cutoff")
ax.fill_between(core_vol.index, 0, core_vol.max(),
                where=(regime=="high_vol"), alpha=0.12, color="red")
ax.legend(); ax.set_title("Market volatility regime (explore)"); plt.tight_layout(); plt.show()

In [ ]:
# gold vs real-yield correlation, split by regime
g = core_ret["gold"]; ry = macro_chg["ust10y_real"]
pair = pd.concat([g, ry, regime.rename("regime")], axis=1).dropna()
pair.columns = ["gold","real_chg","regime"]
by_regime = pair.groupby("regime").apply(
    lambda d: d["gold"].corr(d["real_chg"]), include_groups=False)
print("gold vs real-yield-change correlation, by regime:")
print(by_regime.round(3))
print("\nfull-sample (for reference):", round(pair['gold'].corr(pair['real_chg']),3))

**What to look for:** if the high-vol correlation is markedly more negative
than the calm one, that confirms the relationship is regime-conditional — gold
tracks real yields tightly in stress, loosely in calm. That's the finding that
justifies a regime-aware prediction layer rather than a single global fit.

## Next
- If the regime split is real, `03_gpr_gate.ipynb`: does geopolitical risk (GPR)
  add anything **incremental** over VIX + real yields, especially in the calm
  regime where the yield link is weak?
- Then a modest, regime-aware prediction layer on the pruned basket, evaluated
  on the untouched holdout against a random-walk baseline.